# Construcción de la Master Table (Sprint 1)

Pipeline para procesar las fuentes crudas y consolidar la tabla analítica a nivel de cliente único (`customer_unique_id`).

## 1. Importación de Librerías

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the scripts package importable from this notebook's directory
sys.path.insert(0, str(Path(".").resolve()))

from scripts.data_loader import load_raw_datasets
from scripts.feature_engineering import (
    add_premium_target,
    aggregate_customers,
    build_master_table,
    compute_order_item_features,
    compute_order_payments,
    compute_order_reviews,
    enrich_orders,
    integrate_geo,
    parse_order_dates,
)
from scripts.pipeline import run_pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Configuración de Rutas

In [ ]:
from scripts.config import PROCESSED_DIR, RAW_DIR

print("Project root:", RAW_DIR.parents[1])
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)

## 3. Carga de Datos

In [ ]:
datasets = load_raw_datasets()

customers   = datasets["customers"]
orders      = datasets["orders"]
payments    = datasets["payments"]
reviews     = datasets["reviews"]
order_items = datasets["order_items"]
products    = datasets["products"]
sellers     = datasets["sellers"]

## 4. Dimensiones y Duplicados Crudos

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "payments": payments,
    "reviews": reviews,
    "order_items": order_items,
    "products": products,
    "sellers": sellers,
}

summary = []

for name, df in datasets.items():
    summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicated_rows": df.duplicated().sum(),
        "total_nulls": df.isna().sum().sum(),
    })

pd.DataFrame(summary)


,dataset,rows,columns,duplicated_rows,total_nulls
0,customers,99441,5,0,0
1,orders,99441,8,0,4908
2,payments,103886,5,0,0
3,reviews,99224,7,0,145903
4,order_items,112650,7,0,0
5,products,32951,9,0,2448
6,sellers,3095,4,0,0


## 5. Conversión de Fechas

In [ ]:
orders = parse_order_dates(orders)
orders[["order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"]].head()

## 6. Métricas de Pago por Pedido

In [ ]:
order_payments = compute_order_payments(payments)
order_payments.head()

## 7. Métricas de Artículos por Pedido

In [ ]:
order_item_features = compute_order_item_features(order_items)
order_item_features.head()

## 8. Calificación de Reseñas por Pedido

In [ ]:
order_reviews = compute_order_reviews(reviews)
order_reviews.head()

## 9. Consolidación a Nivel Pedido

In [ ]:
orders_enriched = enrich_orders(orders, customers, order_payments, order_item_features, order_reviews)
orders_enriched.head()

## 10. Agregación a Nivel Cliente

Consolidación de comportamiento de compra general e histórico financiero.

In [ ]:
customer_features = aggregate_customers(orders_enriched)
customer_features.head()

## 11. Integración de Datos Geográficos

In [ ]:
master_table = integrate_geo(customers, customer_features)
master_table.head()

## 12. Generación del Target `is_premium`

Definido preliminarmente en el percentil 80 de gasto neto.

In [ ]:
master_table, premium_threshold = add_premium_target(master_table)

print("Premium threshold (p80):", premium_threshold)
master_table["is_premium"].value_counts(normalize=True).rename("proportion")

## 13. Validación de Calidad de la Master Table

In [13]:
print("Rows:", master_table.shape[0])
print("Columns:", master_table.shape[1])
print("Duplicated customer_unique_id:", master_table["customer_unique_id"].duplicated().sum())
print("Null values in key columns:", master_table[["customer_unique_id", "total_spent", "total_orders"]].isna().sum().sum())
print("Null values total:", master_table.isna().sum().sum())

master_table.describe(include="all").T.head(30)


Rows: 96096
Columns: 28
Duplicated customer_unique_id: 0
Null values in key columns: 0
Null values total: 14411


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
customer_unique_id,96096,96096,206e64e8af2633a2ebe158a7fcb860db,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,96096.0,NaN,NaN,NaN,35184.412463,1003.0,11390.0,24440.0,59032.75,99990.0,29800.101792
customer_city,96096,4119,sao paulo,14971,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,96096,27,SP,40292,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_orders,96096.0,NaN,NaN,NaN,1.034809,1.0,1.0,1.0,1.0,17.0,0.214384
total_items,96096.0,NaN,NaN,NaN,1.172265,0.0,1.0,1.0,1.0,24.0,0.627071
total_products,96096.0,NaN,NaN,NaN,1.065861,0.0,1.0,1.0,1.0,16.0,0.339595
avg_review_score,95380.0,NaN,NaN,NaN,4.084963,1.0,4.0,5.0,5.0,5.0,1.341661
total_reviews,96096.0,NaN,NaN,NaN,1.032551,0.0,1.0,1.0,1.0,17.0,0.269283
first_purchase,96096,NaN,NaN,NaN,2017-12-30 19:19:10.429206016,2016-09-04 21:15:19,2017-09-11 19:52:06,2018-01-18 13:33:08,2018-05-04 10:38:45,2018-10-17 17:30:18,NaN


## 14. Métricas Descriptivas de Negocio

In [14]:
business_metrics = master_table.groupby("is_premium").agg(
    customers=("customer_unique_id", "count"),
    avg_total_spent=("total_spent", "mean"),
    median_total_spent=("total_spent", "median"),
    avg_total_orders=("total_orders", "mean"),
    avg_ticket=("avg_ticket", "mean"),
    avg_review_score=("avg_review_score", "mean"),
    avg_recency_days=("recency_days", "mean"),
)

business_metrics


,customers,avg_total_spent,median_total_spent,avg_total_orders,avg_ticket,avg_review_score,avg_recency_days
is_premium,,,,,,,
0,76874,91.757715,84.14,1.019031,93.986304,4.095153,288.375289
1,19222,435.369844,313.78,1.097909,416.138103,4.044182,285.177765


## 15. Exportación a Parquet

In [ ]:
output_file = PROCESSED_DIR / "master_table.parquet"

master_table.to_parquet(
    output_file,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Master Table saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")

# --- Alternatively: run the full pipeline in one call ---
# master_table = run_pipeline(incremental=False)

## 16. Variables Generadas

In [16]:
master_table.columns.tolist()

['customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'total_orders',
 'total_items',
 'total_products',
 'avg_review_score',
 'total_reviews',
 'first_purchase',
 'last_purchase',
 'avg_delivery_days',
 'avg_estimated_delivery_days',
 'delivered_orders',
 'canceled_orders',
 'late_deliveries',
 'payment_methods_count',
 'main_payment_type',
 'total_spent',
 'avg_ticket',
 'avg_order_price',
 'avg_freight_value',
 'avg_freight_ratio',
 'recency_days',
 'customer_lifetime_days',
 'cancellation_rate',
 'late_delivery_rate',
 'is_premium']